# Elasticsearch Index Operations
## Create, Mapping, Reindex, Analyzers & More

> **Part of:** `cheatsheets` repo  
> **Topic:** Index lifecycle — from creation to reindexing

---

## Table of Contents
1. [Create an Index](#1-create-an-index)
2. [Mappings](#2-mappings)
3. [Analyzers](#3-analyzers)
4. [CRUD Operations](#4-crud-operations)
5. [Search Queries](#5-search-queries)
6. [Reindex](#6-reindex)
7. [Aliases](#7-aliases)
8. [Index Lifecycle](#8-index-lifecycle)
9. [Common Errors & Fixes](#9-common-errors--fixes)

---

## 1. Create an Index

### Basic
```json
PUT my_index
```

### With settings and mapping
```json
PUT my_index
{
  "settings": {
    "number_of_shards": 1,
    "number_of_replicas": 1,
    "analysis": {
      "analyzer": {
        "my_analyzer": {
          "type": "english"
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "title":   { "type": "text" },
      "category":{ "type": "keyword" }
    }
  }
}
```

### Copy mapping from existing index
```json
# Step 1 — get existing mapping
GET blogs/_mapping

# Step 2 — create new index with that mapping
PUT blogs_v2
{
  "mappings": {
    "properties": {
      // paste properties from Step 1
    }
  }
}
```

### Useful index operations
```json
GET my_index            # view index info
GET my_index/_mapping   # view mapping only
GET my_index/_settings  # view settings only
GET my_index/_count     # count documents
DELETE my_index         # delete index ⚠️
HEAD my_index           # check if index exists (200=yes, 404=no)
```

---

## 2. Mappings

### keyword vs text
| | `keyword` | `text` |
|---|---|---|
| Use for | Exact values, IDs, categories | Sentences, paragraphs |
| Searchable | Exact match only | Partial/fuzzy/relevance |
| Aggregatable | ✅ Yes | ❌ No |
| Sortable | ✅ Yes | ❌ No |
| Analysed | ❌ No | ✅ Yes |

### Field type quick reference
| Field | Type | Reason |
|-------|------|--------|
| `first_name`, `last_name` | `keyword` | Exact filter |
| `email`, `url`, `id` | `keyword` | Exact match |
| `category`, `status`, `locale` | `keyword` | Aggregation |
| `title`, `body`, `content` | `text` | Full-text search |
| `description`, `seo` | `text` | Full-text search |
| `title` (search + sort) | `text` + `keyword` | Multi-field |
| `timestamp`, `date` | `date` | Time range |
| `count`, `price` | `integer`/`float` | Numeric range |
| `active`, `published` | `boolean` | True/false |

### Multi-field (text + keyword)
```json
"title": {
  "type": "text",
  "fields": {
    "keyword": {
      "type": "keyword",
      "ignore_above": 256
    }
  }
}
```
```json
# Use title for full-text search
{ "query": { "match": { "title": "happy trees" }}}

# Use title.keyword for exact match / sort / aggregation
{ "query": { "term": { "title.keyword": "Making Happy Little Trees" }}}
```

> **Never search both `title` and `title.keyword` in the same query** — you'll get duplicate results with inflated scores.

### Nested objects
```json
"authors": {
  "properties": {
    "first_name": { "type": "keyword" },
    "last_name":  { "type": "keyword" },
    "company":    { "type": "keyword" },
    "title":      { "type": "text" },
    "job_title": {
      "type": "text",
      "fields": {
        "keyword": { "type": "keyword", "ignore_above": 256 }
      }
    }
  }
}
```

### Dynamic mapping settings
```json
PUT my_index
{
  "mappings": {
    "dynamic": "strict"   // reject unknown fields
    "dynamic": "true"     // auto-map unknown fields (default)
    "dynamic": "false"    // ignore unknown fields silently
  }
}
```

### Add new field to existing mapping
```json
PUT my_index/_mapping
{
  "properties": {
    "new_field": { "type": "keyword" }
  }
}
```
> ⚠️ You **cannot** change an existing field type. You must reindex.

---

## 3. Analyzers

Analyzers control how text is broken down and indexed. They only apply to `text` fields.

### How an analyzer works
```
Input text:  "The Quick Brown FOX jumps!"
      ↓
  Tokenizer:  ["The", "Quick", "Brown", "FOX", "jumps"]
      ↓
  Token filter: lowercase, stop words, stemming
      ↓
  Stored as:  ["quick", "brown", "fox", "jump"]
```

### Built-in analyzers
| Analyzer | What it does |
|----------|-------------|
| `standard` | Default — lowercases, removes punctuation |
| `english` | Standard + removes stop words (the, is, at) + stemming (jumping→jump) |
| `simple` | Splits on non-letters, lowercases |
| `whitespace` | Splits on whitespace only, no lowercasing |
| `keyword` | No analysis — treats entire value as one token |

### English analyzer
```json
PUT blogs
{
  "settings": {
    "analysis": {
      "analyzer": {
        "english_analyzer": {
          "type": "english"
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "content": {
        "type": "text",
        "analyzer": "english"
      }
    }
  }
}
```

**What english analyzer does:**
```
"The fox was jumping over the fence"
         ↓
["fox", "jump", "fenc"]   ← stop words removed, stems extracted

So searching "jumps" will match "jumping", "jumped", "jumps" ✅
```

### Test an analyzer
```json
GET _analyze
{
  "analyzer": "english",
  "text": "The fox was jumping over the fence"
}

# Result:
{
  "tokens": [
    { "token": "fox" },
    { "token": "jump" },
    { "token": "fenc" }
  ]
}
```

### Custom analyzer
```json
PUT my_index
{
  "settings": {
    "analysis": {
      "analyzer": {
        "my_custom_analyzer": {
          "type": "custom",
          "tokenizer": "standard",
          "filter": ["lowercase", "stop", "snowball"]
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "content": {
        "type": "text",
        "analyzer": "my_custom_analyzer"
      }
    }
  }
}
```

### search_analyzer vs analyzer
```json
"content": {
  "type": "text",
  "analyzer": "english",         // used at index time
  "search_analyzer": "standard"  // used at search time
}
```
> Most of the time use the same analyzer for both. Only specify `search_analyzer` if you have a specific reason.

---

## 4. CRUD Operations

### Create / Index documents
```json
# Auto-generate ID
POST my_index/_doc
{ "title": "Hello World", "category": "test" }

# Specify ID
PUT my_index/_doc/1
{ "title": "Hello World", "category": "test" }

# Create only (fail if ID exists)
PUT my_index/_doc/1?op_type=create
{ "title": "Hello World" }
```

### Read
```json
GET my_index/_doc/1          # get by ID
GET my_index/_search         # get all documents
GET my_index/_search?size=50 # get 50 documents
```

### Update
```json
# Partial update (keep other fields)
POST my_index/_update/1
{ "doc": { "category": "updated" }}

# Full replace
PUT my_index/_doc/1
{ "title": "New Title", "category": "new" }
```

### Delete
```json
DELETE my_index/_doc/1    # delete document by ID
DELETE my_index           # delete entire index ⚠️
```

---

## 5. Search Queries

### match vs term
```json
# match → text fields, full-text search
{ "query": { "match": { "content": "happy trees" }}}

# term → keyword fields, exact match
{ "query": { "term": { "category": "Painting" }}}

# terms → keyword, match multiple values
{ "query": { "terms": { "category": ["Painting", "Drawing"] }}}
```

### bool query (combine conditions)
```json
GET my_index/_search
{
  "query": {
    "bool": {
      "must":     [{ "match": { "content": "trees" }}],   // AND - affects score
      "filter":   [{ "term":  { "category": "Painting" }}], // AND - no score
      "should":   [{ "match": { "title": "happy" }}],     // OR - boosts score
      "must_not": [{ "term":  { "published": false }}]    // NOT
    }
  }
}
```

### range query
```json
GET my_index/_search
{
  "query": {
    "range": {
      "publish_date": {
        "gte": "2021-01-01",
        "lte": "2021-12-31"
      }
    }
  }
}
```

### aggregations
```json
GET my_index/_search
{
  "size": 0,
  "aggs": {
    "by_category": {
      "terms": { "field": "category" }
    },
    "posts_over_time": {
      "date_histogram": {
        "field": "publish_date",
        "calendar_interval": "month"
      }
    }
  }
}
```

---

## 6. Reindex

Reindex copies data from one index to another. Use it when you need to change field mappings.

### Why reindex?
```
You CANNOT change field types on existing indices
So the workflow is:
1. Create new index with correct mapping
2. Reindex (copy) data across
3. Use new index going forward
```

### Basic reindex
```json
POST _reindex
{
  "source": { "index": "blogs" },
  "dest":   { "index": "blogs_v2" }
}
```

### Reindex with filter
```json
POST _reindex
{
  "source": {
    "index": "blogs",
    "query": {
      "term": { "category": "Painting" }
    }
  },
  "dest": { "index": "blogs_painting" }
}
```

### Reindex and rename a field
```json
POST _reindex
{
  "source": { "index": "blogs" },
  "dest":   { "index": "blogs_v2" },
  "script": {
    "source": "ctx._source.author_name = ctx._source.remove('author')"
  }
}
```

### Verify reindex
```json
GET blogs/_count        # source count
GET blogs_v2/_count     # dest count — should match
GET blogs_v2/_mapping   # confirm mapping is correct
GET blogs_v2/_search    # spot check documents
```

### Full reindex workflow (Splunk migration pattern)
```json
# Step 1 — check source
GET splunk_logs/_mapping
GET splunk_logs/_count

# Step 2 — create destination with correct mapping
PUT splunk_logs_fixed
{
  "mappings": {
    "properties": {
      "host":      { "type": "keyword" },
      "level":     { "type": "keyword" },
      "message":   { "type": "text", "analyzer": "english" },
      "timestamp": { "type": "date" }
    }
  }
}

# Step 3 — reindex
POST _reindex
{
  "source": { "index": "splunk_logs" },
  "dest":   { "index": "splunk_logs_fixed" }
}

# Step 4 — verify
GET splunk_logs_fixed/_count
GET splunk_logs_fixed/_search

# Step 5 — swap alias (zero downtime)
POST _aliases
{
  "actions": [
    { "remove": { "index": "splunk_logs",       "alias": "logs" }},
    { "add":    { "index": "splunk_logs_fixed",  "alias": "logs" }}
  ]
}
```

---

## 7. Aliases

Aliases let you point a name at an index — useful for zero-downtime reindexing.

```json
# Create alias
POST _aliases
{
  "actions": [
    { "add": { "index": "blogs_v2", "alias": "blogs_current" }}
  ]
}

# Swap alias (atomic — no downtime)
POST _aliases
{
  "actions": [
    { "remove": { "index": "blogs_v1", "alias": "blogs_current" }},
    { "add":    { "index": "blogs_v2", "alias": "blogs_current" }}
  ]
}

# View aliases
GET _aliases
GET blogs_v2/_alias

# Delete alias
DELETE blogs_v2/_alias/blogs_current
```

---

## 8. Index Lifecycle

```
Create index → Index documents → Search/Aggregate
      ↓                               ↓
  Mapping wrong?              Need to change mapping?
      ↓                               ↓
  Create new index            Create new index
  with correct mapping        Reindex data across
      ↓                       Swap alias
  Reindex data                Delete old index
      ↓
  Verify + swap alias
  Delete old index
```

---

## 9. Common Errors & Fixes

| Error | Cause | Fix |
|-------|-------|-----|
| `index_not_found_exception` | Index doesn't exist | Create index first with `PUT` |
| `field type not match` | Wrong query for field type | Use `term` for keyword, `match` for text |
| `mapper_parsing_exception` | Wrong data type for field | Check mapping, fix document |
| `illegal_argument_exception` | Changing existing field type | Reindex to new index |
| Duplicate results | Searching `title` + `title.keyword` | Use only one subfield per query |
| No results | Analyzer mismatch | Test with `GET _analyze` |

---

*Part of personal cheatsheets repo — github.com/longchung90/cheatsheets*



PUT blogs_fixed

GET blogs/_mapping

PUT blogs_fixed/_mapping
{
  "properties": {
    "authors": {
      "properties": {
        "company": {
          "type": "keyword"
        },
        "first_name": {
          "type": "keyword"
        },
        "job_title": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "last_name": {
          "type": "keyword"
        },
        "title": {
          "type": "text"
        }
      }
    },
    "category": {
      "type": "keyword"
    },
    "content": {
      "type": "text"
    },
    "locale": {
      "type": "keyword"
    },
    "publish_date": {
      "type": "date",
      "format": "iso8601"
    },
    "seo": {
      "type": "text"
    },
    "title": {
      "type": "text",
      "fields": {
        "keyword": {
          "type": "keyword",
          "ignore_above": 256
        }
      }
    },
    "url": {
      "type": "keyword"
    },
    "versions": {
      "type": "keyword"
    }
  }
}

# Elasticsearch Index Operations
## Create, Mapping, Reindex, Analyzers & More

> **Part of:** `cheatsheets` repo  
> **Topic:** Index lifecycle — from creation to reindexing

---

## Table of Contents
1. [Create an Index](#1-create-an-index)
2. [Mappings](#2-mappings)
3. [Analyzers](#3-analyzers)
4. [CRUD Operations](#4-crud-operations)
5. [Search Queries](#5-search-queries)
6. [Reindex](#6-reindex)
7. [Aliases](#7-aliases)
8. [Index Lifecycle](#8-index-lifecycle)
9. [Common Errors & Fixes](#9-common-errors--fixes)

---

## 1. Create an Index

### Basic
```json
PUT my_index
```

### With settings and mapping
```json
PUT my_index
{
  "settings": {
    "number_of_shards": 1,
    "number_of_replicas": 1,
    "analysis": {
      "analyzer": {
        "my_analyzer": {
          "type": "english"
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "title":   { "type": "text" },
      "category":{ "type": "keyword" }
    }
  }
}
```

### Copy mapping from existing index
```json
# Step 1 — get existing mapping
GET blogs/_mapping

# Step 2 — create new index with that mapping
PUT blogs_v2
{
  "mappings": {
    "properties": {
      // paste properties from Step 1
    }
  }
}
```

### Useful index operations
```json
GET my_index            # view index info
GET my_index/_mapping   # view mapping only
GET my_index/_settings  # view settings only
GET my_index/_count     # count documents
DELETE my_index         # delete index ⚠️
HEAD my_index           # check if index exists (200=yes, 404=no)
```

---

## 2. Mappings

### keyword vs text
| | `keyword` | `text` |
|---|---|---|
| Use for | Exact values, IDs, categories | Sentences, paragraphs |
| Searchable | Exact match only | Partial/fuzzy/relevance |
| Aggregatable | ✅ Yes | ❌ No |
| Sortable | ✅ Yes | ❌ No |
| Analysed | ❌ No | ✅ Yes |

### Field type quick reference
| Field | Type | Reason |
|-------|------|--------|
| `first_name`, `last_name` | `keyword` | Exact filter |
| `email`, `url`, `id` | `keyword` | Exact match |
| `category`, `status`, `locale` | `keyword` | Aggregation |
| `title`, `body`, `content` | `text` | Full-text search |
| `description`, `seo` | `text` | Full-text search |
| `title` (search + sort) | `text` + `keyword` | Multi-field |
| `timestamp`, `date` | `date` | Time range |
| `count`, `price` | `integer`/`float` | Numeric range |
| `active`, `published` | `boolean` | True/false |

### Multi-field (text + keyword)
```json
"title": {
  "type": "text",
  "fields": {
    "keyword": {
      "type": "keyword",
      "ignore_above": 256
    }
  }
}
```
```json
# Use title for full-text search
{ "query": { "match": { "title": "happy trees" }}}

# Use title.keyword for exact match / sort / aggregation
{ "query": { "term": { "title.keyword": "Making Happy Little Trees" }}}
```

> **Never search both `title` and `title.keyword` in the same query** — you'll get duplicate results with inflated scores.

### Nested objects
```json
"authors": {
  "properties": {
    "first_name": { "type": "keyword" },
    "last_name":  { "type": "keyword" },
    "company":    { "type": "keyword" },
    "title":      { "type": "text" },
    "job_title": {
      "type": "text",
      "fields": {
        "keyword": { "type": "keyword", "ignore_above": 256 }
      }
    }
  }
}
```

### Dynamic mapping settings
```json
PUT my_index
{
  "mappings": {
    "dynamic": "strict"   // reject unknown fields
    "dynamic": "true"     // auto-map unknown fields (default)
    "dynamic": "false"    // ignore unknown fields silently
  }
}
```

### Add new field to existing mapping
```json
PUT my_index/_mapping
{
  "properties": {
    "new_field": { "type": "keyword" }
  }
}
```
> ⚠️ You **cannot** change an existing field type. You must reindex.

---

## 3. Analyzers

Analyzers control how text is broken down and indexed. They only apply to `text` fields.

### How an analyzer works
```
Input text:  "The Quick Brown FOX jumps!"
      ↓
  Tokenizer:  ["The", "Quick", "Brown", "FOX", "jumps"]
      ↓
  Token filter: lowercase, stop words, stemming
      ↓
  Stored as:  ["quick", "brown", "fox", "jump"]
```

### Built-in analyzers
| Analyzer | What it does |
|----------|-------------|
| `standard` | Default — lowercases, removes punctuation |
| `english` | Standard + removes stop words (the, is, at) + stemming (jumping→jump) |
| `simple` | Splits on non-letters, lowercases |
| `whitespace` | Splits on whitespace only, no lowercasing |
| `keyword` | No analysis — treats entire value as one token |

### English analyzer
```json
PUT blogs
{
  "settings": {
    "analysis": {
      "analyzer": {
        "english_analyzer": {
          "type": "english"
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "content": {
        "type": "text",
        "analyzer": "english"
      }
    }
  }
}
```

**What english analyzer does:**
```
"The fox was jumping over the fence"
         ↓
["fox", "jump", "fenc"]   ← stop words removed, stems extracted

So searching "jumps" will match "jumping", "jumped", "jumps" ✅
```

### Test an analyzer
```json
GET _analyze
{
  "analyzer": "english",
  "text": "The fox was jumping over the fence"
}

# Result:
{
  "tokens": [
    { "token": "fox" },
    { "token": "jump" },
    { "token": "fenc" }
  ]
}
```

### Custom analyzer
```json
PUT my_index
{
  "settings": {
    "analysis": {
      "analyzer": {
        "my_custom_analyzer": {
          "type": "custom",
          "tokenizer": "standard",
          "filter": ["lowercase", "stop", "snowball"]
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "content": {
        "type": "text",
        "analyzer": "my_custom_analyzer"
      }
    }
  }
}
```

### search_analyzer vs analyzer
```json
"content": {
  "type": "text",
  "analyzer": "english",         // used at index time
  "search_analyzer": "standard"  // used at search time
}
```
> Most of the time use the same analyzer for both. Only specify `search_analyzer` if you have a specific reason.

---

## 4. CRUD Operations

### Create / Index documents
```json
# Auto-generate ID
POST my_index/_doc
{ "title": "Hello World", "category": "test" }

# Specify ID
PUT my_index/_doc/1
{ "title": "Hello World", "category": "test" }

# Create only (fail if ID exists)
PUT my_index/_doc/1?op_type=create
{ "title": "Hello World" }
```

### Read
```json
GET my_index/_doc/1          # get by ID
GET my_index/_search         # get all documents
GET my_index/_search?size=50 # get 50 documents
```

### Update
```json
# Partial update (keep other fields)
POST my_index/_update/1
{ "doc": { "category": "updated" }}

# Full replace
PUT my_index/_doc/1
{ "title": "New Title", "category": "new" }
```

### Delete
```json
DELETE my_index/_doc/1    # delete document by ID
DELETE my_index           # delete entire index ⚠️
```

---

## 5. Search Queries

### match vs term
```json
# match → text fields, full-text search
{ "query": { "match": { "content": "happy trees" }}}

# term → keyword fields, exact match
{ "query": { "term": { "category": "Painting" }}}

# terms → keyword, match multiple values
{ "query": { "terms": { "category": ["Painting", "Drawing"] }}}
```

### bool query (combine conditions)
```json
GET my_index/_search
{
  "query": {
    "bool": {
      "must":     [{ "match": { "content": "trees" }}],   // AND - affects score
      "filter":   [{ "term":  { "category": "Painting" }}], // AND - no score
      "should":   [{ "match": { "title": "happy" }}],     // OR - boosts score
      "must_not": [{ "term":  { "published": false }}]    // NOT
    }
  }
}
```

### range query
```json
GET my_index/_search
{
  "query": {
    "range": {
      "publish_date": {
        "gte": "2021-01-01",
        "lte": "2021-12-31"
      }
    }
  }
}
```

### aggregations
```json
GET my_index/_search
{
  "size": 0,
  "aggs": {
    "by_category": {
      "terms": { "field": "category" }
    },
    "posts_over_time": {
      "date_histogram": {
        "field": "publish_date",
        "calendar_interval": "month"
      }
    }
  }
}
```

---

## 6. Reindex

Reindex copies data from one index to another. Use it when you need to change field mappings.

### Why reindex?
```
You CANNOT change field types on existing indices
So the workflow is:
1. Create new index with correct mapping
2. Reindex (copy) data across
3. Use new index going forward
```

### Basic reindex
```json
POST _reindex
{
  "source": { "index": "blogs" },
  "dest":   { "index": "blogs_v2" }
}
```

### Reindex with filter
```json
POST _reindex
{
  "source": {
    "index": "blogs",
    "query": {
      "term": { "category": "Painting" }
    }
  },
  "dest": { "index": "blogs_painting" }
}
```

### Reindex and rename a field
```json
POST _reindex
{
  "source": { "index": "blogs" },
  "dest":   { "index": "blogs_v2" },
  "script": {
    "source": "ctx._source.author_name = ctx._source.remove('author')"
  }
}
```

### Verify reindex
```json
GET blogs/_count        # source count
GET blogs_v2/_count     # dest count — should match
GET blogs_v2/_mapping   # confirm mapping is correct
GET blogs_v2/_search    # spot check documents
```

### Full reindex workflow (Splunk migration pattern)
```json
# Step 1 — check source
GET splunk_logs/_mapping
GET splunk_logs/_count

# Step 2 — create destination with correct mapping
PUT splunk_logs_fixed
{
  "mappings": {
    "properties": {
      "host":      { "type": "keyword" },
      "level":     { "type": "keyword" },
      "message":   { "type": "text", "analyzer": "english" },
      "timestamp": { "type": "date" }
    }
  }
}

# Step 3 — reindex
POST _reindex
{
  "source": { "index": "splunk_logs" },
  "dest":   { "index": "splunk_logs_fixed" }
}

# Step 4 — verify
GET splunk_logs_fixed/_count
GET splunk_logs_fixed/_search

# Step 5 — swap alias (zero downtime)
POST _aliases
{
  "actions": [
    { "remove": { "index": "splunk_logs",       "alias": "logs" }},
    { "add":    { "index": "splunk_logs_fixed",  "alias": "logs" }}
  ]
}
```

---

## 7. Aliases

Aliases let you point a name at an index — useful for zero-downtime reindexing.

```json
# Create alias
POST _aliases
{
  "actions": [
    { "add": { "index": "blogs_v2", "alias": "blogs_current" }}
  ]
}

# Swap alias (atomic — no downtime)
POST _aliases
{
  "actions": [
    { "remove": { "index": "blogs_v1", "alias": "blogs_current" }},
    { "add":    { "index": "blogs_v2", "alias": "blogs_current" }}
  ]
}

# View aliases
GET _aliases
GET blogs_v2/_alias

# Delete alias
DELETE blogs_v2/_alias/blogs_current
```

---

## 8. Index Lifecycle

```
Create index → Index documents → Search/Aggregate
      ↓                               ↓
  Mapping wrong?              Need to change mapping?
      ↓                               ↓
  Create new index            Create new index
  with correct mapping        Reindex data across
      ↓                       Swap alias
  Reindex data                Delete old index
      ↓
  Verify + swap alias
  Delete old index
```

---

## 9. Common Errors & Fixes

| Error | Cause | Fix |
|-------|-------|-----|
| `index_not_found_exception` | Index doesn't exist | Create index first with `PUT` |
| `field type not match` | Wrong query for field type | Use `term` for keyword, `match` for text |
| `mapper_parsing_exception` | Wrong data type for field | Check mapping, fix document |
| `illegal_argument_exception` | Changing existing field type | Reindex to new index |
| Duplicate results | Searching `title` + `title.keyword` | Use only one subfield per query |
| No results | Analyzer mismatch | Test with `GET _analyze` |

---

*Part of personal cheatsheets repo — github.com/longchung90/cheatsheets*